In [ ]:
# INSTALL LIBRARIES

!pip install -q --upgrade transformers datasets evaluate rouge_score sentencepiece gradio accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00


In [ ]:
# IMPORT LIBRARIES

import re
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import pipeline
import evaluate
import gradio as gr

In [ ]:
# CHECK DEVICE   - Runs faster on GPU

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

Using device: GPU


In [ ]:
# LOAD DATASET

dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)
print(dataset["train"][0].keys())
print("\nSample article:\n", dataset["train"][0]["article"][:500])
print("\nSample summary:\n", dataset["train"][0]["highlights"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
dict_keys(['article', 'highlights', 'id'])

Sample article:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

Sample summary:
 Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor 

In [ ]:
# SET SUBSETS

train_raw = dataset["train"].select(range(3000))
val_raw = dataset["validation"].select(range(300))
test_raw = dataset["test"].select(range(300))

print("Train size:", len(train_raw))
print("Validation size:", len(val_raw))
print("Test size:", len(test_raw))

Train size: 3000
Validation size: 300
Test size: 300


In [ ]:
# CLEAN TEXT

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
# EXAMPLES

def make_pipeline_example(article, summary, input_ratio=0.35, min_words=40):
    article = clean_text(article)
    summary = clean_text(summary)

    words = article.split()
    if len(words) < min_words:
        return None

    split_index = max(12, int(len(words) * input_ratio))
    prompt = " ".join(words[:split_index])
    reference_completion = " ".join(words[split_index:])

    return {
        "prompt": prompt,
        "reference_completion": reference_completion,
        "full_text": article,
        "reference_summary": summary
    }

In [ ]:
# PREPROCESS DATASET SPLITS

def preprocess_split(split):
    processed = []
    for item in split:
        ex = make_pipeline_example(item["article"], item["highlights"])
        if ex is not None:
            processed.append(ex)
    return processed

train_data = preprocess_split(train_raw)
val_data = preprocess_split(val_raw)
test_data = preprocess_split(test_raw)

print("Processed train:", len(train_data))
print("Processed val:", len(val_data))
print("Processed test:", len(test_data))

print("\nExample keys:", train_data[0].keys())
print("\nPrompt sample:\n", train_data[0]["prompt"][:400])
print("\nReference summary:\n", train_data[0]["reference_summary"])

Processed train: 2997
Processed val: 300
Processed test: 300

Example keys: dict_keys(['prompt', 'reference_completion', 'full_text', 'reference_summary'])

Prompt sample:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his ca

Reference summary:
 Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor says he has no plans to fritter his cash away . Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
# SAVE PROCESSED DATA

with open("train_data.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open("val_data.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open("test_data.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print("Processed files saved.")

Processed files saved.


In [ ]:
# LOAD PRETRAINED MODELS

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch # Ensure torch is imported for device check

autocomplete_model = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

# Using AutoTokenizer and AutoModelForSeq2SeqLM for summarization
summarizer_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

# Define device here to ensure it's available and consistent
summarizer_device = "cuda:0" if torch.cuda.is_available() else "cpu"

summarizer_model_core = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(summarizer_device)

print(f"Summarizer model type: {type(summarizer_model_core)}")
print(f"Summarizer model device: {summarizer_model_core.device}")

print("Models loaded successfully.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Summarizer model type: <class 'transformers.models.bart.modeling_bart.BartForConditionalGeneration'>
Summarizer model device: cuda:0
Models loaded successfully.


In [ ]:
# AUTOCOMPLETE FUNCTION

def postprocess_generated_text(text):
    text = clean_text(text)

    # Remove anything after a quote (cuts fake quotes)
    if '"' in text:
        text = text.split('"')[0]

    # Trim to last full sentence
    last_punct = max(text.rfind("."), text.rfind("!"), text.rfind("?"))
    if last_punct > 0:
        text = text[:last_punct + 1]

    return clean_text(text)


def autocomplete_text(prompt, max_new_tokens=35):
    result = autocomplete_model(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # NO randomness
        num_beams=4,              # more stable output
        no_repeat_ngram_size=3,   # reduce repetition
        repetition_penalty=1.2,
        early_stopping=True,
        pad_token_id=50256
    )

    generated_text = result[0]["generated_text"]
    return postprocess_generated_text(generated_text)

In [ ]:
# SUMMARISE FUNCTION

def summarize_text(text, max_length=60, min_length=15):
    text = clean_text(text)

    if len(text.split()) < 30:
        return "Text too short to summarise properly."

    inputs = summarizer_tokenizer([text], max_length=1024, return_tensors="pt", truncation=True)
    inputs = {k: v.to(summarizer_model_core.device) for k, v in inputs.items()}

    summary_ids = summarizer_model_core.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True
    )

    summary = [
        summarizer_tokenizer.decode(
            g,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        for g in summary_ids
    ]

    return clean_text(summary[0])

In [ ]:
# FULL PIPELINE FUNCTION

def full_pipeline(prompt):
    prompt = clean_text(prompt)
    completed_text = autocomplete_text(prompt, max_new_tokens=35)
    summary = summarize_text(completed_text, max_length=50, min_length=12)
    return completed_text, summary

In [ ]:
# TEST PIPELINE

sample_prompt = test_data[0]["prompt"]

completed_text, summary = full_pipeline(sample_prompt)

print("INPUT PROMPT:\n")
print(sample_prompt)

print("\n" + "="*80)
print("AUTOCOMPLETE OUTPUT:\n")
print(completed_text)

print("\n" + "="*80)
print("GENERATED SUMMARY:\n")
print(summary)

print("\n" + "="*80)
print("REFERENCE SUMMARY:\n")
print(test_data[0]["reference_summary"])

[transformers] Passing `generation_config` together with generation-related arguments=({'early_stopping', 'do_sample', 'no_repeat_ngram_size', 'pad_token_id', 'repetition_penalty', 'max_new_tokens', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_wi

INPUT PROMPT:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday

In [ ]:
# ROUGE EVALUATION

rouge = evaluate.load("rouge")

In [ ]:
# EVALUATE SUMMARY

predictions = []
references = []

evaluation_samples = test_data[:20]  # keep small for speed

for i, item in enumerate(evaluation_samples):
    try:
        completed_text, generated_summary = full_pipeline(item["prompt"])
        predictions.append(generated_summary)
        references.append(item["reference_summary"])
        print(f"Processed sample {i+1}/{len(evaluation_samples)}")
    except Exception as e:
        print(f"Error on sample {i+1}: {e}")

[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed sample 1/20


[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed sample 2/20


[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Processed sample 17/20


[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed sample 18/20


[transformers] Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed sample 19/20
Processed sample 20/20


In [ ]:
# ROUGE RESULTS

results = rouge.compute(predictions=predictions, references=references)

print("ROUGE Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

ROUGE Evaluation Results:
rouge1: 0.2301
rouge2: 0.0757
rougeL: 0.1666
rougeLsum: 0.1696


The higher ROUGE-1 score shows that the model captures key words and main
topics, while the lower ROUGE-2 score indicates weaker phrase-level alignment,
meaning the model does not always match exact wording.
ROUGE-L reflects partial structural similarity, but not strong consistency.

In [ ]:
# EVALUATION TABLE

rows = []

for i in range(min(5, len(evaluation_samples))):
    rows.append({
        "Prompt": evaluation_samples[i]["prompt"][:150],
        "Generated Summary": predictions[i] if i < len(predictions) else "",
        "Reference Summary": references[i] if i < len(references) else ""
    })

df = pd.DataFrame(rows)
df

,Prompt,Generated Summary,Reference Summary
0,(CNN)The Palestinian Authority officially beca...,The Palestinian Authority becomes the 123rd me...,Membership gives the ICC jurisdiction over all...
1,(CNN)Never mind cats having nine lives. A stra...,"The dog was hit by a car, apparently whacked o...","Theia, a bully breed mix, was apparently hit b..."
2,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif is the Iranian foreign mi...,Mohammad Javad Zarif has spent more time with ...
3,(CNN)Five Americans who were monitored for thr...,One of the five had a heart-related issue on S...,17 Americans were exposed to the Ebola virus w...
4,(CNN)A Duke student has admitted to hanging a ...,A Duke student has admitted to hanging a noose...,Student is no longer on Duke University campus...


In [ ]:
# INTERACTIVE FUNCTION

def demo_pipeline(user_input):
    completed_text, summary = full_pipeline(user_input)
    return completed_text, summary

In [ ]:
# GRADIO INTERFACE

demo = gr.Interface(
    fn=demo_pipeline,
    inputs=gr.Textbox(
        lines=6,
        placeholder="Enter the beginning of a sentence or paragraph here..."
    ),
    outputs=[
        gr.Textbox(lines=10, label="Autocompleted Text"),
        gr.Textbox(lines=5, label="Generated Summary")
    ],
    title="Sentence Autocomplete and Summarisation Pipeline",
    description="Enter context text. The system completes the text and then generates a summary."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://152565bcf6112be39b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Sample Prompts:

Main: Researchers at a British university have developed a new artificial intelligence system that can help doctors detect early signs of disease from routine medical scans. The team said the model performed well in early trials and could reduce diagnosis times in hospitals. Lead researchers said the system may


Secondary: A leading renewable energy company has unveiled plans to expand its solar and wind power operations across several regions. The project is expected to create thousands of jobs and significantly reduce carbon emissions over the next decade. Company representatives said the initiative reflects growing demand for sustainable energy solutions and could


Additional: A leading international company has reported strong financial results for the past quarter, driven by increased demand for its products and services. The company’s executives said they are optimistic about future growth, citing expansion into new markets and continued investment in innovation. However, analysts noted that ongoing economic uncertainty could